In [1]:
import os
import sys
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import confusion_matrix, f1_score, classification_report
import numpy as np
PROJECT_ROOT = os.path.abspath("..") 
sys.path.append(PROJECT_ROOT)

In [ ]:

from dataset.otto_final import TraceOttoDataset
from model.trace import TRACE
from utils.SplitData import split_data_Train_Val_Test
from utils.feature_engineering import get_between_features, get_elapsed_feature
from utils.plot_confussion_matrix import plot_confusion_matrix
from utils.training_utils import update_binary_metrics
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [ ]:
trace_model = TRACE(
    num_embeddings_aid=1855603,
    num_embeddings_event_type=4,
    embedding_dim=32,
    num_classes=3
).to(device)

checkpoint = torch.load("Model_TRACE_ATC_FinalVersion_SingleTask.pt", weights_only=False, map_location=device)

trace_model.load_state_dict(checkpoint["model_state_dict"])
best_thr_ATC = checkpoint["best_global_threshold"]["ATC"]
best_thr_SAT = checkpoint["best_global_threshold"]["SAT"]
best_thr_MAP = checkpoint["best_global_threshold"]["MAP"]

trace_model.eval()
print(f"Loaded model Optimum threshold ATC: {best_thr_ATC:.4f} SAT: {best_thr_SAT:.4f}, SAT: {best_thr_MAP:.4f}")

In [ ]:
dataset_processed  = TraceOttoDataset(
    file_name='../train.jsonl',
    input_seq_len=64,
    min_timestamps_per_sample=32,
)

In [ ]:
train_loader, validation_loader, test_loader = split_data_Train_Val_Test(dataset_processed, batch_size=32)


print(len(train_loader.dataset))
print(len(validation_loader.dataset))
print(len(test_loader.dataset))


In [ ]:
trace_model.eval()
criterion = torch.nn.BCEWithLogitsLoss()
test_loss = 0.0
test_correct_ATC = 0
test_total_ATC = 0
test_correct_SAT = 0
test_total_SAT = 0
test_correct_MAP = 0
test_total_MAP = 0

y_true_ATC = []
y_pred_ATC = []
y_true_SAT = []
y_pred_SAT = []
y_true_MAP = []
y_pred_MAP = []
with torch.no_grad():
    for inputs_test, targets_test in test_loader:
        target_test_task_ATC = targets_test["ATC"].unsqueeze(1).to(device).float()
        target_test_task_SAT = targets_test["SAT"].unsqueeze(1).to(device).float()
        target_test_task_MAP = targets_test["MAP"].unsqueeze(1).to(device).float()
        
        inputs_test = {k: v.to(device) for k, v in inputs_test.items()}

        delta_elapsed = get_elapsed_feature(inputs_test["timestamps"]).to(device).float()
        delta_between = get_between_features(inputs_test["timestamps"]).to(device).float()

        logits = trace_model(
            inputs_test["aid"],
            inputs_test["type"],
            delta_elapsed,
            delta_between
        )

        loss_ATC = criterion(logits[:, 0:1], target_test_task_ATC)
        loss_SAT = criterion(logits[:, 1:2], target_test_task_SAT)
        loss_MAP = criterion(logits[:, 2:3], target_test_task_MAP)
        
        loss_test = (loss_ATC + loss_SAT + loss_MAP)
        test_loss += loss_test.item()
        
        test_correct_ATC, test_total_ATC = update_binary_metrics(
            logits[:, 0:1],
            target_test_task_ATC,
            test_correct_ATC,
            test_total_ATC,
            y_true_ATC,
            y_pred_ATC,
            threshold=best_thr_ATC
        )

        test_correct_SAT, test_total_SAT = update_binary_metrics(
            logits[:, 1:2],
            target_test_task_SAT,
            test_correct_SAT,
            test_total_SAT,
            y_true_SAT,
            y_pred_SAT,
            threshold=best_thr_SAT
        )
        test_correct_MAP, test_total_MAP = update_binary_metrics(
            logits[:, 2:3],
            target_test_task_MAP,
            test_correct_MAP,
            test_total_MAP,
            y_true_MAP,
            y_pred_MAP,
            threshold=best_thr_MAP
        )
        

test_loss /= len(test_loader)
test_accuracy_ATC = test_correct_ATC / max(test_total_ATC, 1)
test_accuracy_SAT = test_correct_SAT / max(test_total_SAT, 1)
test_accuracy_MAP = test_correct_MAP / max(test_total_MAP, 1)


y_true_ATC = torch.cat(y_true_ATC).numpy().astype(int).ravel()
y_pred_ATC = torch.cat(y_pred_ATC).numpy().astype(int).ravel()

y_true_SAT = torch.cat(y_true_SAT).numpy().astype(int).ravel()
y_pred_SAT = torch.cat(y_pred_SAT).numpy().astype(int).ravel()

y_true_MAP = torch.cat(y_true_MAP).numpy().astype(int).ravel()
y_pred_MAP = torch.cat(y_pred_MAP).numpy().astype(int).ravel()

f1_ATC = f1_score(y_true_ATC, y_pred_ATC, zero_division=0)
f1_SAT = f1_score(y_true_SAT, y_pred_SAT, zero_division=0)
f1_MAP = f1_score(y_true_MAP, y_pred_MAP, zero_division=0)
macro_f1_ATC = f1_score(y_true_ATC, y_pred_ATC, zero_division=0, average="macro")
macro_f1_SAT = f1_score(y_true_SAT, y_pred_SAT, zero_division=0, average="macro")
macro_f1_MAP = f1_score(y_true_MAP, y_pred_MAP, zero_division=0, average="macro")


f1_mean = (f1_ATC + f1_SAT + f1_MAP) / 3
cm_ATC = confusion_matrix(y_true_ATC, y_pred_ATC)
cm_SAT = confusion_matrix(y_true_SAT, y_pred_SAT)
cm_MAP = confusion_matrix(y_true_MAP, y_pred_MAP)

test_accuracy_global = (test_correct_ATC + test_correct_SAT + test_correct_MAP) / max(test_total_ATC + test_total_SAT + test_total_MAP, 1)

print(f"Test accuracy global: {test_accuracy_global:.4f}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {np.mean([test_accuracy_ATC, test_accuracy_SAT,test_accuracy_MAP]):.4f}")
print(f"F1 Mean Score: {f1_mean}")
print(f"F1 ATC Score: {f1_ATC}")
print(f"F1 SAT Score: {f1_SAT}")
print(f"F1 MAP Score: {f1_MAP}")
print(f"Macro F1 ATC Score: {macro_f1_ATC}")
print(f"Macro F1 SAT Score: {macro_f1_SAT}")
print(f"Macro F1 MAP Score: {macro_f1_MAP}")
print(f"Optimum Threshold ATC: {best_thr_ATC:.4f}")
print(f"Optimum Threshold SAT: {best_thr_SAT:.4f}")
print(f"Optimum Threshold MAP: {best_thr_MAP:.4f}")



print("\nClassification Report ATC:")
print(classification_report(y_true_ATC, y_pred_ATC, target_names=["ATC 0", "ATC 1"]))

print("\nClassification Report SAT:")
print(classification_report(y_true_SAT, y_pred_SAT, target_names=["SAT 0", "SAT 1"]))

print("\nClassification Report MAP:")
print(classification_report(y_true_MAP, y_pred_MAP, target_names=["MAP 0", "MAP 1"]))

print("Confusion Matrix: ATC\n", cm_ATC)
fig1 = plot_confusion_matrix(
    cm_ATC,
    name_task="ATC",
    name_classes=["ATC 0", "ATC 1"]
)
print("Confusion Matrix: SAT\n", cm_SAT)
fig2 = plot_confusion_matrix(
    cm_SAT,
    name_task="SAT",
    name_classes=["SAT 0", "SAT 1"]
)
fig3 = plot_confusion_matrix(
    cm_MAP,
    name_task="MAP",
    name_classes=["MAP 0", "MAP 1"]
)
plt.show()
plt.close(fig1)
plt.close(fig2)
plt.close(fig3)
